# task_08 Solution

In [ ]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_08/data/")


In [ ]:

students = pd.read_csv(base / "students.csv")
courses = pd.read_csv(base / "courses.csv")
enrollments = pd.read_csv(base / "enrollments.csv")
attendance = pd.read_csv(base / "attendance.csv")
activities = pd.read_csv(base / "activities.csv")

Q1: For each student, compute a credit-weighted exam average and multiply it by the student's overall attendance rate. Restrict to students who took at least one STEM course and at least one Humanities course and whose monthly attendance rate was never below 85%. Which homeroom has the highest average attendance-penalized score across those students? If there is a tie, use the smallest homeroom label.

In [3]:
students = pd.read_csv(base / "students.csv")
attendance = pd.read_csv(base / "attendance.csv")
enrollments = pd.read_csv(base / "enrollments.csv")
courses = pd.read_csv(base / "courses.csv")

attendance["monthly_attendance_rate"] = (attendance["school_days"] - attendance["absent_days"]) / attendance["school_days"]
worst_month = attendance.groupby("student_id")["monthly_attendance_rate"].min().rename("worst_month_rate")
overall_attendance = attendance.assign(attended_days=attendance["school_days"] - attendance["absent_days"]).groupby("student_id").apply(lambda g: g["attended_days"].sum() / g["school_days"].sum(), include_groups=False).rename("overall_attendance_rate")

enroll = enrollments.merge(courses, on="course_id")
weighted_exam = enroll.groupby("student_id").apply(lambda g: (g["exam_score"] * g["credit"]).sum() / g["credit"].sum(), include_groups=False).rename("weighted_exam")
subjects = enroll.groupby("student_id")["subject"].agg(lambda s: set(s))

student_scores = pd.concat([weighted_exam, overall_attendance, worst_month], axis=1).reset_index().merge(students, on="student_id")
student_scores["has_both_subject_groups"] = student_scores["student_id"].map(lambda sid: {"STEM", "Humanities"}.issubset(subjects[sid]))
eligible = student_scores[(student_scores["has_both_subject_groups"]) & (student_scores["worst_month_rate"] >= 0.85)].copy()
eligible["attendance_penalized_score"] = eligible["weighted_exam"] * eligible["overall_attendance_rate"]

q1 = str(eligible.groupby("homeroom")["attendance_penalized_score"].mean().sort_values(ascending=False).index[0])
print(q1)

HR5


Q4: Among students enrolled in at least 4 courses, compute each student's credit-weighted homework-exam gap. Which student_id has the largest?

In [ ]:
# Merge enrollments with courses for credit info
enroll_c = enrollments.merge(courses[["course_id", "credit"]], on="course_id")

# Filter to students with >= 4 courses
course_counts = enroll_c.groupby("student_id").size()
eligible = course_counts[course_counts >= 4].index
enroll_elig = enroll_c[enroll_c["student_id"].isin(eligible)].copy()

# Compute credit-weighted gap: sum((hw - exam) * credit) / sum(credit)
enroll_elig["weighted_gap"] = (enroll_elig["homework_score"] - enroll_elig["exam_score"]) * enroll_elig["credit"]
student_gap = enroll_elig.groupby("student_id").apply(
    lambda g: g["weighted_gap"].sum() / g["credit"].sum(), include_groups=False
).rename("cw_gap")

q4 = student_gap.sort_values(ascending=False).index[0]
print(q4)